# Mobile Money Agent Network: Data Audit and Clean-Up

**Analyst:** Timothy Kerre | **Cohort:** 1milliondevs 2026 | **Assignment:** PLP Data Analytics A3, Engagement 2

**Scenario:** A mobile-money provider wants to know which agents and counties handle the most deposits and withdrawals. The legacy transaction extract is unreliable, so it is audited, cleaned and merged with the agent register before any analysis. All cleaning rules live in `cleaning_pipeline.py` as one function, `clean_agent_data()`, which also returns a cleaning log for the data governance team.

## Setup and load the data

**Approach:** The raw CSVs are not committed to the repository. If they are missing, `generate_data.py` recreates them. In Colab, the repository is cloned first so that `cleaning_pipeline.py` can be imported.

In [1]:
import os
import subprocess
import sys

# In Colab only the notebook is opened, so fetch the repository for the helper files.
if not os.path.exists("cleaning_pipeline.py"):
    subprocess.run(["git", "clone", "https://github.com/kerre-star/plp-da-a3-duka-sales-and-agent-cleanup.git"], check=True)
    os.chdir("plp-da-a3-duka-sales-and-agent-cleanup")

# Regenerate the raw data if it is not already present.
if not (os.path.exists("agents.csv") and os.path.exists("agent_transactions.csv")):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

import pandas as pd
from cleaning_pipeline import clean_agent_data

agents = pd.read_csv("agents.csv", parse_dates=["onboarded"])
txns = pd.read_csv("agent_transactions.csv")

print("agents.csv            :", agents.shape)
print("agent_transactions.csv:", txns.shape)

agents.csv            : (20, 3)
agent_transactions.csv: (612, 5)


## First look at the raw data

**Approach:** Shape, data types, missing values, category labels and the date range are checked before any cleaning. Note that `pd.read_csv` silently converts the text `"N/A"` to `NaN`, so the missing-value count below mixes blank cells and `"N/A"` entries. They are separated in the audit.

In [2]:
txns.info()
print()
print("Missing values per column:")
print(txns.isna().sum())
print()
print("txn_type labels:")
print(txns["txn_type"].value_counts(dropna=False))
print()
print("Distinct agent_id values in transactions:", txns["agent_id"].nunique(),
      "| in the agent register:", agents["agent_id"].nunique())
print("Agents per county:")
print(agents["county"].value_counts())
txns.head()

<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    str    
 1   agent_id  612 non-null    str    
 2   txn_type  612 non-null    str    
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    str    
dtypes: float64(1), str(4)
memory usage: 24.0 KB

Missing values per column:
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64

txn_type labels:
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64

Distinct agent_id values in transactions: 21 | in the agent register: 20
Agents per county:
county
Nairobi    6
Mombasa    5
Kisumu     4
Nakuru     3
Kiambu     2
Name: count, dtype: int64


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


## Task 1: Data-quality audit

**Approach:** The raw file is read again with every column as text and `keep_default_na=False`. That separates blank amounts from `"N/A"` text, which pandas would otherwise merge into `NaN`. Each issue is counted on the raw file (612 rows) and classified under one of six data-quality dimensions: Completeness, Uniqueness, Validity, Consistency, Accuracy and Timeliness. The result is stored as a DataFrame.

In [3]:
raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

# IQR fence for amounts, using each transaction once
unique_raw = raw.drop_duplicates("txn_id")
amount_num = pd.to_numeric(unique_raw["amount"], errors="coerce")
q1, q3 = amount_num.quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)

# Transactions dated before the agent was onboarded (timeliness check)
txn_dates = pd.to_datetime(raw["txn_date"].str.replace("/", "-", regex=False), format="%Y-%m-%d")
onboarded = raw["agent_id"].map(agents.set_index("agent_id")["onboarded"])

audit_rows = [
    ("Repeated txn_id (12 extra copies of existing transactions)", "Uniqueness", "txn_id",
     raw["txn_id"].duplicated().sum()),
    ("txn_type spelt in mixed case or with trailing space", "Consistency", "txn_type",
     (~raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("Blank amount", "Completeness", "amount",
     (raw["amount"].str.strip() == "").sum()),
    ("'N/A' text in amount (pandas silently reads it as NaN)", "Validity", "amount",
     (raw["amount"].str.strip() == "N/A").sum()),
    ("txn_date written with '/' instead of '-'", "Validity", "txn_date",
     raw["txn_date"].str.contains("/", regex=False).sum()),
    ("Amount of exactly 250,000 (suspected cap or placeholder)", "Accuracy", "amount",
     (raw["amount"] == "250000").sum()),
    (f"Other amounts above the IQR fence of {upper_fence:,.1f}", "Accuracy", "amount",
     ((amount_num > upper_fence) & (amount_num != 250000)).sum()),
    ("agent_id 'AG099' is not in the agent register", "Consistency", "agent_id",
     (~raw["agent_id"].isin(agents["agent_id"])).sum()),
    ("Transaction dated before the agent was onboarded", "Timeliness", "txn_date",
     (txn_dates < onboarded).sum()),
]

audit = pd.DataFrame(audit_rows, columns=["issue", "dimension", "column", "rows_affected"])
audit["rows_affected"] = audit["rows_affected"].astype(int)
audit

,issue,dimension,column,rows_affected
0,Repeated txn_id (12 extra copies of existing t...,Uniqueness,txn_id,12
1,txn_type spelt in mixed case or with trailing ...,Consistency,txn_type,384
2,Blank amount,Completeness,amount,15
3,'N/A' text in amount (pandas silently reads it...,Validity,amount,7
4,txn_date written with '/' instead of '-',Validity,txn_date,5
5,"Amount of exactly 250,000 (suspected cap or pl...",Accuracy,amount,5
6,"Other amounts above the IQR fence of 6,084.5",Accuracy,amount,35
7,agent_id 'AG099' is not in the agent register,Consistency,agent_id,8
8,Transaction dated before the agent was onboarded,Timeliness,txn_date,0


**Audit findings:** All injected problems were found, and each dimension is covered. The raw file has 612 rows, of which 12 are repeated `txn_id` values. The 7 `"N/A"` amounts include 2 that sit on duplicated rows, so there are 5 distinct transactions with `"N/A"`. The timeliness check found nothing: no transaction predates its agent's onboarding, and the extract covers a single window from 1 March to 29 April 2026.

## Run the cleaning pipeline (Task 9)

**Approach:** All cleaning rules are in `clean_agent_data(txns, agents)` in `cleaning_pipeline.py`. It uses no hard-coded row numbers or manual edits, and returns the clean DataFrame together with a log. The function is called once here, and the sections for Tasks 2-7 show what each step did.

In [4]:
clean_df, log = clean_agent_data(txns, agents)

print(f"Rows before cleaning: {len(txns):,}  |  rows after cleaning: {len(clean_df):,}")
log

Rows before cleaning: 612  |  rows after cleaning: 600


,step,dimension,issue,rows_affected,action
0,0,Summary,Rows in 612 -> rows out 600,12,Only exact txn_id duplicates were removed
1,1,Uniqueness,Duplicate txn_id rows,12,"Dropped repeated txn_id values, keeping the fi..."
2,2,Consistency,txn_type not already 'Deposit' or 'Withdrawal',376,"Stripped spaces, ignored case and mapped to De..."
3,2,Consistency,txn_type values that could not be mapped,0,None expected; any would be left as missing
4,3,Validity,txn_date written with '/' instead of '-',5,Replaced '/' with '-' and parsed as datetime
5,3,Validity,txn_date that could not be parsed,0,None expected; any would be left as NaT
6,4,Completeness,"Missing amount (blank, or 'N/A' read as NaN by...",20,Flagged for imputation
7,4,Validity,Non-numeric amount text,0,Coerced to missing and flagged for imputation
8,4,Validity,Zero or negative amount,0,Treated as invalid and flagged for imputation
9,4,Completeness,Amounts imputed with the median for the txn_type,20,Filled and flagged in amount_imputed


## Task 2: Uniqueness

**Approach:** Duplicates are identified by `txn_id`, not by comparing whole rows. A whole-row comparison would miss a duplicate whose other fields differ.

In [5]:
removed = len(txns) - clean_df["txn_id"].nunique()
print("Duplicate transactions removed:", removed)
print("Duplicate txn_id values left in clean data:", clean_df["txn_id"].duplicated().sum())
print()
print("The one duplicate pair whose copies differ (agent_id):")
raw[raw["txn_id"] == "TX00405"]

Duplicate transactions removed: 12
Duplicate txn_id values left in clean data: 0

The one duplicate pair whose copies differ (agent_id):


,txn_id,agent_id,txn_type,amount,txn_date
404,TX00405,AG014,Deposit,1936.0,2026-04-03
601,TX00405,AG099,Deposit,1936.0,2026-04-03


**Answer:** 12 duplicate transactions were removed (612 to 600 rows). 11 are exact copies. The 12th, `TX00405`, has two copies that differ only in `agent_id` (`AG014` and `AG099`). The first occurrence was kept, which is the one with the registered agent, so it is not counted as an unknown-agent transaction.

## Task 3: Consistency of txn_type

**Approach:** Values are stripped of spaces, lower-cased and mapped to `Deposit` or `Withdrawal`. The raw `value_counts()` shows five spellings (`deposit`, `Deposit`, `DEPOSIT `, `withdrawal`, `Withdrawal`).

In [6]:
print("Before (raw):")
print(txns["txn_type"].value_counts(), "\n")
print("After (clean):")
print(clean_df["txn_type"].value_counts(dropna=False))

Before (raw):
txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64 

After (clean):
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64


**Answer:** `txn_type` now holds exactly two values, `Deposit` (373 transactions) and `Withdrawal` (227). No value failed to map. The pipeline logs 376 of the 600 unique transactions as changed, which includes the trailing-space variant `"DEPOSIT "` that a plain `.str.lower()` would have missed.

## Task 4: Validity and data types

**Approach:** `txn_date` is parsed with an explicit `%Y-%m-%d` format after replacing `/` with `-`. `amount` is converted with `pd.to_numeric(errors="coerce")`, so any non-numeric text becomes missing and is handled in Task 5. Because pandas already turns `"N/A"` into `NaN` on loading, missing amounts are found with `isna()` as well as by checking for invalid strings.

In [7]:
print(clean_df.dtypes)
print()
print("Date range:", clean_df["txn_date"].min().date(), "to", clean_df["txn_date"].max().date())
print("Unparseable dates:", clean_df["txn_date"].isna().sum())
print("Amounts that are missing after cleaning:", clean_df["amount"].isna().sum())

txn_id                         str
agent_id                       str
txn_type                       str
amount                     float64
txn_date            datetime64[us]
amount_imputed                bool
is_outlier                    bool
county                         str
agent_registered              bool
dtype: object

Date range: 2026-03-01 to 2026-04-29
Unparseable dates: 0
Amounts that are missing after cleaning: 0


**Answer:** Five dates written as `2026/03/05`-style strings were parsed along with the ISO ones, and none failed. `amount` is numeric (`float64`) and `txn_date` is `datetime64`. All 600 transactions fall between 1 March and 29 April 2026.

## Task 5: Completeness (missing and invalid amounts)

**Decision:** The 20 transactions without a usable amount (15 blank and 5 `"N/A"`) are kept and given the **median amount of their transaction type**, with `amount_imputed = True`.

**Why:**
- **Not dropping them:** they are valid transactions (3.3% of the data), and dropping them would understate transaction counts per agent and county.
- **Median, not mean:** amounts are strongly right-skewed and include the 250,000 values, which would pull a mean far above a typical transaction.
- **By transaction type:** deposit and withdrawal medians differ slightly.
- **Flagged:** the flag lets any analysis exclude imputed values.

In [8]:
medians = clean_df.loc[~clean_df["amount_imputed"]].groupby("txn_type")["amount"].median()
imputed_total = clean_df.loc[clean_df["amount_imputed"], "amount"].sum()

print("Median used to fill each type:")
print(medians, "\n")
print("Transactions imputed:", clean_df["amount_imputed"].sum())
print(f"Imputed value: KES {imputed_total:,.0f} "
      f"({imputed_total / clean_df['amount'].sum():.1%} of total value)")

Median used to fill each type:
txn_type
Deposit       1803.5
Withdrawal    1750.5
Name: amount, dtype: float64 

Transactions imputed: 20
Imputed value: KES 35,699 (1.3% of total value)


**Answer:** 20 amounts were imputed (Deposit median KES 1,803.50, Withdrawal median KES 1,750.50). Together they add KES 35,699, which is 1.3% of total value, so the imputation has very little effect on the totals.

## Task 6: Outliers (IQR method)

**Approach:** The IQR fences are calculated from observed (non-imputed) amounts: `Q1 - 1.5 x IQR` and `Q3 + 1.5 x IQR`. Amounts outside the fences are flagged in `is_outlier`. Nothing is removed. Then the 250,000 values are investigated separately, because the IQR flag cannot tell a natural high value from a suspicious one.

In [9]:
observed = clean_df.loc[~clean_df["amount_imputed"], "amount"]
q1, q3 = observed.quantile([0.25, 0.75])
iqr = q3 - q1
print(f"Q1 = {q1:,.0f}   Q3 = {q3:,.0f}   IQR = {iqr:,.0f}")
print(f"Fences: {q1 - 1.5 * iqr:,.1f} to {q3 + 1.5 * iqr:,.1f}")
print("Flagged as outliers:", clean_df["is_outlier"].sum())

suspect = clean_df["amount"].eq(250_000)
natural = clean_df["is_outlier"] & ~suspect
print("  exactly KES 250,000:", suspect.sum())
print("  other flagged amounts:", natural.sum(), f"(largest KES {clean_df.loc[natural, 'amount'].max():,.0f})")
print()
print(f"Median amount: KES {clean_df['amount'].median():,.0f}")
print(f"250,000 is {250_000 / clean_df['amount'].median():,.0f}x the median and "
      f"{250_000 / clean_df.loc[natural, 'amount'].max():.1f}x the largest other flagged amount")
print(f"The five values are {clean_df.loc[suspect, 'amount'].sum() / clean_df['amount'].sum():.1%} "
      f"of total value (KES {clean_df.loc[suspect, 'amount'].sum():,.0f} of KES {clean_df['amount'].sum():,.0f})")
print()
clean_df.loc[suspect, ["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date"]]

Q1 = 1,077   Q3 = 3,080   IQR = 2,003
Fences: -1,927.5 to 6,084.5
Flagged as outliers: 40
  exactly KES 250,000: 5
  other flagged amounts: 35 (largest KES 28,847)

Median amount: KES 1,791
250,000 is 140x the median and 8.7x the largest other flagged amount
The five values are 46.5% of total value (KES 1,250,000 of KES 2,688,596)



,txn_id,agent_id,county,txn_type,amount,txn_date
306,TX00307,AG012,Kisumu,Deposit,250000.0,2026-03-02
315,TX00316,AG014,Nairobi,Deposit,250000.0,2026-04-01
424,TX00425,AG005,Mombasa,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Nakuru,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Kiambu,Deposit,250000.0,2026-03-11


**Answer and recommendation:** The IQR fences flag 40 transactions. 35 of them are between KES 6,085 and KES 28,847, a normal long tail for money-transfer amounts. The other 5 are all exactly KES 250,000. That is about 140 times the median and 8.7 times the next-largest value, and they make up 46.5% of total value. One falls in each of the five counties, on four different days (two on 2 March), so this is not one agent's activity.

The business should **keep them in the data but not trust them** until each is checked against the source ledger, since a round figure can be a system limit, a placeholder or a keying error. Until then, reports should show totals with and without them, as Task 8 does. They should not be deleted automatically, because a genuine KES 250,000 transaction is valid and relevant to risk monitoring.

## Task 7: Merge with the agent register

**Approach:** A **left join** from transactions to the register on `agent_id` keeps every transaction. An inner join would silently drop transactions from unregistered agents. The join is validated as many-to-one, and the row counts are checked before and after.

In [10]:
unique_rows = txns["txn_id"].nunique()
unknown = clean_df[~clean_df["agent_registered"]]

print("Unique transactions before merge:", unique_rows)
print("Rows after merge               :", len(clean_df))
print("Rows an inner join would keep  :", clean_df["agent_registered"].sum())
print()
print("Transactions with unknown agents:", len(unknown), "| agent_id values:", unknown["agent_id"].unique())
print(f"Their value: KES {unknown['amount'].sum():,.0f} ({unknown['amount'].sum() / clean_df['amount'].sum():.1%} of total)")
print()
unknown[["txn_id", "agent_id", "txn_type", "amount", "txn_date", "county"]]

Unique transactions before merge: 600
Rows after merge               : 600
Rows an inner join would keep  : 593

Transactions with unknown agents: 7 | agent_id values: <StringArray>
['AG099']
Length: 1, dtype: str
Their value: KES 27,156 (1.0% of total)



,txn_id,agent_id,txn_type,amount,txn_date,county
55,TX00056,AG099,Deposit,8507.0,2026-04-01,Unknown
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,Unknown
77,TX00078,AG099,Deposit,1770.0,2026-04-06,Unknown
237,TX00238,AG099,Deposit,4944.0,2026-04-06,Unknown
267,TX00268,AG099,Deposit,3110.0,2026-04-09,Unknown
444,TX00445,AG099,Deposit,5309.0,2026-04-27,Unknown
557,TX00558,AG099,Deposit,1639.0,2026-03-22,Unknown


**Answer:** 7 of the 600 transactions (1.2%, KES 27,156) reference `AG099`, which is not in the register. (The raw file had 8, but one was the duplicate copy of `TX00405` removed in Task 2.) They were **kept** with county `Unknown` and `agent_registered = False`, so total values reconcile with the cleaned data. They appear as a separate `Unknown` row in the summaries and should be referred to the agent-onboarding team to establish whether `AG099` is a new agent missing from the register or an invalid code.

## Task 8: Summary tables

**Approach:** (a) The county summary groups by county and transaction type, with total value, number of transactions and average value. (b) The weekly pivot sums transaction value by week and county. Weeks run Monday to Sunday and are labelled by their start date. Because 1 March 2026 is a Sunday, the first week holds one day only, and the last week (starting 27 April) holds three. Each table is shown with all valid data and, as a sensitivity check, without the five KES 250,000 transactions.

In [11]:
by_county_type = (
    clean_df.groupby(["county", "txn_type"])
    .agg(total_value=("amount", "sum"),
         transactions=("txn_id", "count"),
         avg_value=("amount", "mean"))
    .round(0)
)
by_county_type

total_value  transactions  avg_value
county  txn_type                                        
Kiambu  Deposit        334737.0            43     7785.0
        Withdrawal      77024.0            27     2853.0
Kisumu  Deposit        436502.0            74     5899.0
        Withdrawal      76348.0            37     2063.0
Mombasa Deposit        149094.0            72     2071.0
        Withdrawal     368592.0            61     6042.0
Nairobi Deposit        543698.0           118     4608.0
        Withdrawal     161932.0            62     2612.0
Nakuru  Deposit        184944.0            60     3082.0
        Withdrawal     328568.0            39     8425.0
Unknown Deposit         25279.0             6     4213.0
        Withdrawal       1877.0             1     1877.0

In [12]:
county_total = (
    clean_df.groupby("county")
    .agg(total_value=("amount", "sum"), transactions=("txn_id", "count"))
    .sort_values("total_value", ascending=False)
)
county_total["share_pct"] = (county_total["total_value"] / county_total["total_value"].sum() * 100).round(1)
county_total

,total_value,transactions,share_pct
county,,,
Nairobi,705630.0,180,26.2
Mombasa,517686.0,133,19.3
Nakuru,513512.5,99,19.1
Kisumu,512850.0,111,19.1
Kiambu,411761.5,70,15.3
Unknown,27156.0,7,1.0


In [13]:
# Sensitivity check: the same county summary without the five KES 250,000 transactions
no_suspect = clean_df[~suspect]

by_county_type_excl = (
    no_suspect.groupby(["county", "txn_type"])
    .agg(total_value=("amount", "sum"),
         transactions=("txn_id", "count"),
         avg_value=("amount", "mean"))
    .round(0)
)
print("County and type summary WITHOUT the KES 250,000 transactions:")
display(by_county_type_excl)

county_excl = (
    no_suspect.groupby("county")
    .agg(total_value=("amount", "sum"), transactions=("txn_id", "count"))
    .sort_values("total_value", ascending=False)
)
county_excl["share_pct"] = (county_excl["total_value"] / county_excl["total_value"].sum() * 100).round(1)
county_excl["agents"] = county_excl.index.map(agents["county"].value_counts()).fillna(0).astype(int)
county_excl["value_per_agent"] = (county_excl["total_value"] / county_excl["agents"].where(county_excl["agents"] > 0)).round(0)
county_excl

County and type summary WITHOUT the KES 250,000 transactions:


total_value  transactions  avg_value
county  txn_type                                        
Kiambu  Deposit         84737.0            42     2018.0
        Withdrawal      77024.0            27     2853.0
Kisumu  Deposit        186502.0            73     2555.0
        Withdrawal      76348.0            37     2063.0
Mombasa Deposit        149094.0            72     2071.0
        Withdrawal     118592.0            60     1977.0
Nairobi Deposit        293698.0           117     2510.0
        Withdrawal     161932.0            62     2612.0
Nakuru  Deposit        184944.0            60     3082.0
        Withdrawal      78568.0            38     2068.0
Unknown Deposit         25279.0             6     4213.0
        Withdrawal       1877.0             1     1877.0

,total_value,transactions,share_pct,agents,value_per_agent
county,,,,,
Nairobi,455630.0,179,31.7,6,75938.0
Mombasa,267686.0,132,18.6,5,53537.0
Nakuru,263512.5,98,18.3,3,87838.0
Kisumu,262850.0,110,18.3,4,65712.0
Kiambu,161761.5,69,11.2,2,80881.0
Unknown,27156.0,7,1.9,0,NaN


In [14]:
clean_df["week_start"] = clean_df["txn_date"].dt.to_period("W").dt.start_time.dt.date

weekly = clean_df.pivot_table(index="week_start", columns="county", values="amount",
                              aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
weekly.round(0)

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_start,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2770.0,0.0,18468.0
2026-03-02,18317.0,287933.0,285696.0,68967.0,56574.0,0.0,717488.0
2026-03-09,267746.0,23186.0,30504.0,52888.0,28730.0,0.0,403054.0
2026-03-16,22054.0,28126.0,25172.0,43134.0,39795.0,1639.0,159920.0
2026-03-23,11504.0,27077.0,17050.0,41400.0,18972.0,0.0,116004.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06,15030.0,15341.0,47120.0,56814.0,276609.0,9824.0,420737.0
2026-04-13,12265.0,29470.0,26730.0,56406.0,26180.0,0.0,151051.0
2026-04-20,24507.0,37439.0,16687.0,65135.0,30862.0,1877.0,176507.0


**Findings:**

- **Nairobi handles the most value.** It processed KES 705,630 across 180 transactions (26.2% of the total). Without the 250,000 values its lead is clearer: KES 455,630, or 31.7%, against KES 267,686 for Mombasa, the next county.
- **The 250,000 values distort the picture.** Kiambu's deposit average of KES 7,785 is inflated by one 250,000 deposit. Without it, the average is KES 2,018 and Kiambu's share of value falls from 15.3% to 11.2%.
- **Per agent, the order changes.** Nairobi has the most agents (6 of 20) and the most value, but Nakuru's three agents average KES 87,838 each against Nairobi's KES 75,938, and Kiambu's two average KES 80,881. Mombasa is lowest at KES 53,537 per agent.
- **Deposits outnumber withdrawals** in every county: 373 deposits against 227 withdrawals overall. Nairobi's deposits are KES 543,698 against KES 161,932 for withdrawals.
- **Weekly value is dominated by the 250,000 values.** The week of 2 March totals KES 717,488 because two of them fall in it (Kisumu and Mombasa). The four full weeks without one (16 March, 23 March, 13 April and 20 April) total between KES 116,004 and KES 176,507.

## Save the clean data and run final checks

**Approach:** The cleaned DataFrame is saved to `outputs/agent_transactions_clean.csv` with `index=False`. It is then read back and checked against the requirements: no duplicate `txn_id`, only two `txn_type` values, a numeric `amount`, ISO dates, the two flag columns and the merged `county` column.

In [15]:
os.makedirs("outputs", exist_ok=True)
clean_df.drop(columns="week_start").to_csv("outputs/agent_transactions_clean.csv", index=False)

check = pd.read_csv("outputs/agent_transactions_clean.csv")
assert check["txn_id"].is_unique
assert set(check["txn_type"]) == {"Deposit", "Withdrawal"}
assert pd.api.types.is_numeric_dtype(check["amount"]) and check["amount"].notna().all()
assert check["txn_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
assert {"amount_imputed", "is_outlier", "county"} <= set(check.columns)
assert len(check) == 600

print("All checks passed. Columns:", list(check.columns))
print("Rows:", len(check))
check.head()

All checks passed. Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county', 'agent_registered']
Rows: 600


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,agent_registered
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,True
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,True
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,True
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,True
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,True


## Data Quality Report

**Summary:** The raw extract had **612 rows**. After cleaning it has **600 rows**, with only the 12 repeated transactions removed. Every other issue was corrected, imputed or flagged, so no valid transaction was lost. The cleaned file is `outputs/agent_transactions_clean.csv`, and every step is logged by `clean_agent_data()`. Rows below are counted on the raw file unless stated otherwise.

| Dimension | Issues found | Treatment |
|---|---|---|
| **Completeness** | 20 transactions with no amount: 15 blank and 5 `"N/A"` (after removing duplicates). | Imputed with the median for the transaction type (Deposit KES 1,803.50, Withdrawal KES 1,750.50) and flagged in `amount_imputed`. This adds KES 35,699, or 1.3% of value. |
| **Uniqueness** | 12 repeated `txn_id` values; 11 exact copies and `TX00405`, whose copies differ in `agent_id`. | Removed, keeping the first occurrence (612 to 600 rows). |
| **Validity** | 5 dates written with `/` and 7 `"N/A"` amount entries (5 distinct transactions). No unparseable dates and no zero or negative amounts. | Dates parsed with an explicit format. Non-numeric amounts converted to missing, then imputed. |
| **Consistency** | `txn_type` had 5 spellings (384 raw rows non-standard). 8 raw rows (7 after de-duplication) use `AG099`, which is not in the agent register. | `txn_type` mapped to `Deposit` and `Withdrawal`. Unknown-agent transactions kept (KES 27,156, 1.0% of value), labelled `Unknown`, and referred to onboarding. |
| **Accuracy** | 40 amounts outside the IQR fences (-1,927.5 to 6,084.5): 35 natural high values and 5 that are exactly KES 250,000, which are 46.5% of total value. | Flagged in `is_outlier`, not removed. The 250,000 values should be verified against the source ledger, and reports should show results with and without them. |
| **Timeliness** | None found. No transaction predates its agent's onboarding. | The extract covers only 60 days (1 March to 29 April 2026), so seasonal patterns cannot be assessed, and the first and last weekly figures are partial weeks. |

**Privacy:** This extract holds no customer phone numbers, but if it did, the phone number would be a direct identifier and therefore personal data under the Kenya Data Protection Act 2019, as would any customer name or ID. Combining a phone number with `txn_date`, `amount` and `agent_id` would also reveal where and when an identifiable person transacted, so the transaction details become personal data once linked to a customer. If the agents are individuals, their identifiers and onboarding details would also count. To protect these fields under the Act, I would apply data minimisation (not extracting phone numbers unless the analysis needs them), replace them with salted hashes or tokens before analysis, encrypt the extract in storage and transfer, and restrict access by role and log it. I would also set a retention period for the extract, confirm the purpose for processing and that the provider is registered with the Office of the Data Protection Commissioner, and have a breach-response plan, including notifying the Commissioner within 72 hours of a breach.